# Gauteng Service-Delivery Gaps — Multi-Label Pipeline
### Zindi / Google hackathon · author: Tumo Olorato Mogame

Predict **5 binary service-gap labels** per household — water, sanitation, refuse, energy, education.

**Official metric (weighted-average F1):**
```
Score = 0.3304·F1(Water) + 0.2329·F1(Refuse) + 0.2119·F1(Education) + 0.1200·F1(Energy) + 0.1048·F1(Sanitation)
```
Priority of effort follows the weights: **Water > Refuse > Education > Energy > Sanitation**.

**Pipeline (config-driven, swap one name → rerun):**
Setup → Ingestion → Cleaning → EDA → Metric → Features+Model-Zoo → CV engine →
Baseline sweep → Tuning → Over/under-fit diagnostics → Threshold tuning (val) →
Champion + Ensemble → Submission (+format check) → Submission log.

**Key data facts driving the design** (see EDA): only ~825 unique feature profiles → heavy
irreducible label noise → favour *well-regularised* models + *per-label threshold tuning*, the
single highest-leverage step. `test` contains a `pop_group` category (`Coloured`) unseen in
`train`, so all encoding uses `handle_unknown='ignore'`.

> Only **25 submissions total** — we trust local `val.csv` weighted-F1, not leaderboard probing.
> `val.csv` is used **only** for threshold tuning + local scoring; we never train on it.


## 1 · Setup — installs, imports, reproducibility

In [ ]:
# Extra libs not pre-installed in Colab. Safe to re-run.
!pip install -q iterative-stratification catboost lightgbm xgboost optuna 2>/dev/null
print("deps ready")

In [ ]:
import os, json, random, warnings, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputClassifier, ClassifierChain
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              HistGradientBoostingClassifier)
from sklearn.metrics import f1_score
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

# Optional gradient-boosting libs (guarded so the notebook still runs if one is missing)
HAVE = {}
try:    from xgboost import XGBClassifier;      HAVE['xgb']=True
except Exception: HAVE['xgb']=False
try:    from lightgbm import LGBMClassifier;    HAVE['lgbm']=True
except Exception: HAVE['lgbm']=False
try:    from catboost import CatBoostClassifier;HAVE['catboost']=True
except Exception: HAVE['catboost']=False
print("optional models available:", HAVE)

In [ ]:
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); os.environ['PYTHONHASHSEED']=str(seed)
seed_everything()
print("seed set:", SEED)

## 2 · Config — one place to control everything
Change `DATA_DIR` to wherever the CSVs live. In Colab: mount Drive **or** upload the 5 files.

In [ ]:
# ---- locate data ----
IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    pass

# Candidate locations — first that contains train.csv wins. Edit/extend as needed.
CANDIDATES = [
    'data',
    '.',
    '/content',
    '/content/drive/MyDrive/zindi_service_gap',
    r'C:\\Users\\tumom\\Downloads\\zindi_data',   # local dev
]
DATA_DIR = next((p for p in CANDIDATES if os.path.exists(os.path.join(p,'train.csv'))), None)
assert DATA_DIR, "Could not find train.csv — set DATA_DIR manually or upload the files."
print("DATA_DIR =", DATA_DIR)

# Where to persist models / logs (Drive if available, else local)
ART_DIR = '/content/drive/MyDrive/zindi_service_gap/artifacts' if IN_COLAB else os.path.join(DATA_DIR,'artifacts')
os.makedirs(ART_DIR, exist_ok=True)

In [ ]:
CONFIG = dict(
    seed       = SEED,
    id_col     = 'ID',
    labels     = ['no_water_access','no_sanitation_access','no_refuse_access',
                  'no_energy_access','no_education_access'],
    # Official competition weights (must sum to 1.0)
    weights    = {'no_water_access':0.3304, 'no_refuse_access':0.2329,
                  'no_education_access':0.2119, 'no_energy_access':0.1200,
                  'no_sanitation_access':0.1048},
    n_splits   = 5,
    thr_grid   = np.round(np.linspace(0.05, 0.95, 91), 3),
)
LABELS  = CONFIG['labels']
ID_COL  = CONFIG['id_col']
assert abs(sum(CONFIG['weights'].values()) - 1.0) < 1e-6
print("config ready")

## 3 · Ingestion — load & schema check

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR,'train.csv'))
val   = pd.read_csv(os.path.join(DATA_DIR,'val.csv'))
test  = pd.read_csv(os.path.join(DATA_DIR,'test.csv'))
sample_sub = pd.read_csv(os.path.join(DATA_DIR,'SampleSubmission.csv'))

FEATURES = [c for c in train.columns if c not in [ID_COL]+LABELS]
print(f"train {train.shape} | val {val.shape} | test {test.shape}")
print(f"{len(FEATURES)} feature columns")
assert set(FEATURES).issubset(test.columns), "feature mismatch train/test"
assert list(sample_sub.columns) == [ID_COL]+LABELS, "submission column mismatch"
print("schema OK")

## 4 · Cleaning
All 25 features are low-cardinality categoricals. Findings: **no missing values** except
`age_band == '__SUPPRESSED__'` (~0.3%), which we keep as its own legitimate category.
Nothing to impute or scale — we cast to string so the encoder treats every value as categorical
and `handle_unknown='ignore'` absorbs unseen categories (e.g. `pop_group='Coloured'` in test).

In [ ]:
for df in (train, val, test):
    df[FEATURES] = df[FEATURES].astype(str)

# Sanity: which test categories are unseen in train? (must be handled, not crash)
unseen = {c: sorted(set(test[c]) - set(train[c])) for c in FEATURES}
unseen = {k:v for k,v in unseen.items() if v}
print("unseen test categories:", unseen if unseen else "none")
print("train NaNs:", int(train[FEATURES].isna().sum().sum()))

## 5 · EDA — prevalence, co-occurrence, and the noise ceiling

In [ ]:
# Label prevalence vs weight
print(f"{'label':24s}{'train +rate':>12}{'val +rate':>11}{'weight':>9}")
for l in LABELS:
    print(f"{l:24s}{train[l].mean():>12.3f}{val[l].mean():>11.3f}{CONFIG['weights'][l]:>9}")

print("\n# gaps per household (train):")
print(train[LABELS].sum(1).value_counts().sort_index().to_string())

In [ ]:
# THE key diagnostic: identical feature profiles often carry different labels.
# The per-profile majority-vote accuracy is the ceiling any model can reach.
g = train.groupby(FEATURES)
print("unique feature profiles:", g.ngroups, "of", len(train), "rows")
print("\n# noise ceiling (max accuracy if we perfectly memorised every profile):")
for l in LABELS:
    maj = g[l].transform(lambda s: s.mode().iloc[0])
    print(f"  {l:24s} {(train[l]==maj).mean():.3f}")
print("\n=> heavy irreducible noise. Regularise hard; the win is threshold tuning, not capacity.")

In [ ]:
# Label co-occurrence heat-map
corr = train[LABELS].corr().values
fig,ax=plt.subplots(figsize=(6,5))
im=ax.imshow(corr,cmap='RdYlGn',vmin=-.1,vmax=.5)
sh=['Water','Sanit','Refuse','Energy','Educ']
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels(sh); ax.set_yticklabels(sh)
for i in range(5):
    for j in range(5): ax.text(j,i,f"{corr[i,j]:.2f}",ha='center',va='center',fontsize=8)
ax.set_title("Label co-occurrence (Pearson r)"); plt.colorbar(im,fraction=.046); plt.tight_layout(); plt.show()
print("Water<->Sanitation/Refuse/Energy correlate => ClassifierChain may help.")

## 6 · The official metric (the heart of everything)
The starter notebook uses **macro** F1 — that is *not* the leaderboard metric. We implement the
exact **weighted** average and use it for every model-selection decision. Because the score is a
weighted sum of independent per-label F1s, **tuning each label's threshold separately is optimal**
(a label's threshold only affects its own term).

In [ ]:
def competition_score(y_true, y_pred, weights=CONFIG['weights'], labels=LABELS):
    """Exact Zindi weighted-average F1. Returns (overall, {label: f1})."""
    y_true=np.asarray(y_true); y_pred=np.asarray(y_pred)
    per={}; total=0.0
    for j,l in enumerate(labels):
        f=f1_score(y_true[:,j], y_pred[:,j], zero_division=0)
        per[l]=f; total+=weights[l]*f
    return total, per

def tune_thresholds(y_true, proba, grid=CONFIG['thr_grid']):
    """Per-label F1-maximising threshold on the given (y_true, proba)."""
    y_true=np.asarray(y_true); thr=np.empty(y_true.shape[1])
    for j in range(y_true.shape[1]):
        f1s=[f1_score(y_true[:,j],(proba[:,j]>=t).astype(int),zero_division=0) for t in grid]
        thr[j]=grid[int(np.argmax(f1s))]
    return thr

def apply_thr(proba, thr): return (np.asarray(proba) >= np.asarray(thr)).astype(int)
print("metric ready")

## 7 · Feature pipeline + Model Zoo
One-hot encoding (with `handle_unknown='ignore'`) lives **inside** the Pipeline so it is fit per
CV fold → no leakage. The zoo lets you swap a model by name. Each model can be wrapped two ways:
- **ovr** — one independent classifier per label (`MultiOutputClassifier`)
- **chain** — `ClassifierChain`, feeds each prediction forward to exploit label correlations

In [ ]:
def make_preprocessor():
    return ColumnTransformer(
        [('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), FEATURES)],
        remainder='drop')   # dense: HistGB & some libs reject sparse; matrix is tiny (~84 cols)

def base_estimator(name, seed=SEED):
    if name=='logreg':
        return LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced')
    if name=='rf':
        return RandomForestClassifier(n_estimators=400, min_samples_leaf=3,
                 class_weight='balanced', n_jobs=-1, random_state=seed)
    if name=='extratrees':
        return ExtraTreesClassifier(n_estimators=500, min_samples_leaf=3,
                 class_weight='balanced', n_jobs=-1, random_state=seed)
    if name=='histgb':
        return HistGradientBoostingClassifier(learning_rate=0.06, max_iter=400,
                 l2_regularization=1.0, random_state=seed)
    if name=='xgb' and HAVE['xgb']:
        return XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=4,
                 subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
                 eval_metric='logloss', tree_method='hist', random_state=seed, n_jobs=-1)
    if name=='lgbm' and HAVE['lgbm']:
        return LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=31,
                 subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
                 class_weight='balanced', random_state=seed, n_jobs=-1, verbose=-1)
    if name=='catboost' and HAVE['catboost']:
        return CatBoostClassifier(iterations=500, learning_rate=0.05, depth=5,
                 l2_leaf_reg=3.0, random_seed=seed, verbose=0)
    raise ValueError(f"unknown/unavailable model: {name}")

def make_model(name, strategy='ovr', seed=SEED):
    head = (MultiOutputClassifier(base_estimator(name,seed)) if strategy=='ovr'
            else ClassifierChain(base_estimator(name,seed), order='random', random_state=seed))
    return Pipeline([('pre', make_preprocessor()), ('clf', head)])

def proba_matrix(model, X):
    """Normalise predict_proba to (n_samples, n_labels) of P(gap=1)."""
    p = model.predict_proba(X)
    if isinstance(p, list):                 # MultiOutputClassifier
        return np.column_stack([pi[:,1] for pi in p])
    return np.asarray(p)                     # ClassifierChain already (n, n_labels)
print("model zoo ready")

## 8 · Cross-validation engine
Plain `StratifiedKFold` can't stratify 5 labels at once → we use **MultilabelStratifiedKFold**.
For each candidate we compute **out-of-fold (OOF) probabilities**, tune thresholds on the OOF,
and report the weighted-F1. We also record the **train-fold score** so the train↔CV gap exposes
overfitting (Section 11).

In [ ]:
X_train = train[FEATURES]; y_train = train[LABELS].values.astype(int)
X_val   = val[FEATURES];   y_val   = val[LABELS].values.astype(int)
X_test  = test[FEATURES]

def cv_evaluate(name, strategy='ovr', n_splits=CONFIG['n_splits'], verbose=False):
    mskf = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof = np.zeros((len(X_train), len(LABELS)))
    train_scores=[]; t0=time.time()
    for k,(tr,va) in enumerate(mskf.split(X_train, y_train)):
        m = make_model(name, strategy)
        m.fit(X_train.iloc[tr], y_train[tr])
        oof[va] = proba_matrix(m, X_train.iloc[va])
        ptr = proba_matrix(m, X_train.iloc[tr])
        thr_tr = tune_thresholds(y_train[tr], ptr)
        train_scores.append(competition_score(y_train[tr], apply_thr(ptr, thr_tr))[0])
    thr = tune_thresholds(y_train, oof)
    cv_score, per = competition_score(y_train, apply_thr(oof, thr))
    res = dict(name=name, strategy=strategy, cv=cv_score, train=float(np.mean(train_scores)),
               gap=float(np.mean(train_scores))-cv_score, thr=thr.tolist(), per=per,
               secs=round(time.time()-t0,1), oof=oof)
    if verbose:
        print(f"{name:10s}/{strategy:5s} CV={cv_score:.4f} train={res['train']:.4f} "
              f"gap={res['gap']:.3f} ({res['secs']}s)")
    return res
print("cv engine ready")

## 9 · Baseline sweep — every model at default params
Run the zoo, rank by CV weighted-F1. This tells us which families fit *before* we spend time
tuning. (Drop entries from `ZOO` to speed up.)

In [ ]:
ZOO = [('logreg','ovr'), ('rf','ovr'), ('extratrees','ovr'), ('histgb','ovr'),
       ('logreg','chain'), ('rf','chain')]
if HAVE['xgb']:      ZOO += [('xgb','ovr'), ('xgb','chain')]
if HAVE['lgbm']:     ZOO += [('lgbm','ovr'), ('lgbm','chain')]
if HAVE['catboost']: ZOO += [('catboost','ovr'), ('catboost','chain')]

results = [cv_evaluate(n,s,verbose=True) for n,s in ZOO]
board = (pd.DataFrame([{k:r[k] for k in ['name','strategy','cv','train','gap','secs']}
                       for r in results]).sort_values('cv', ascending=False).reset_index(drop=True))
print("\n=== BASELINE LEADERBOARD (CV weighted-F1) ===")
print(board.to_string(index=False))

## 10 · Hyperparameter tuning (Optuna)
Promote the top model family and search its params, optimising the **exact CV weighted-F1**.
Optuna (Bayesian) beats grid/random for the same budget and is open-source (rules-compliant).

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

TUNE_MODEL = board.iloc[0]['name']      # or hard-set e.g. 'lgbm'
TUNE_STRAT = board.iloc[0]['strategy']
N_TRIALS   = 30                         # raise for a deeper search
print("tuning:", TUNE_MODEL, TUNE_STRAT)

def suggest(trial, name):
    if name=='logreg':
        return dict(C=trial.suggest_float('C',1e-3,100,log=True))
    if name=='histgb':
        return dict(learning_rate=trial.suggest_float('learning_rate',0.01,0.2,log=True),
            max_iter=trial.suggest_int('max_iter',200,800),
            max_leaf_nodes=trial.suggest_int('max_leaf_nodes',15,63),
            min_samples_leaf=trial.suggest_int('min_samples_leaf',5,50),
            l2_regularization=trial.suggest_float('l2_regularization',1e-3,10,log=True))
    if name in ('rf','extratrees'):
        return dict(n_estimators=trial.suggest_int('n_estimators',200,800),
            min_samples_leaf=trial.suggest_int('min_samples_leaf',1,10),
            max_features=trial.suggest_float('max_features',0.3,1.0))
    if name=='lgbm':
        return dict(n_estimators=trial.suggest_int('n_estimators',300,900),
            learning_rate=trial.suggest_float('learning_rate',0.01,0.1,log=True),
            num_leaves=trial.suggest_int('num_leaves',15,63),
            min_child_samples=trial.suggest_int('min_child_samples',5,60),
            subsample=trial.suggest_float('subsample',0.6,1.0),
            colsample_bytree=trial.suggest_float('colsample_bytree',0.6,1.0),
            reg_lambda=trial.suggest_float('reg_lambda',1e-2,10,log=True))
    if name=='xgb':
        return dict(n_estimators=trial.suggest_int('n_estimators',300,900),
            learning_rate=trial.suggest_float('learning_rate',0.01,0.1,log=True),
            max_depth=trial.suggest_int('max_depth',3,7),
            min_child_weight=trial.suggest_int('min_child_weight',1,10),
            subsample=trial.suggest_float('subsample',0.6,1.0),
            colsample_bytree=trial.suggest_float('colsample_bytree',0.6,1.0),
            reg_lambda=trial.suggest_float('reg_lambda',1e-2,10,log=True))
    if name=='catboost':
        return dict(iterations=trial.suggest_int('iterations',300,900),
            learning_rate=trial.suggest_float('learning_rate',0.01,0.1,log=True),
            depth=trial.suggest_int('depth',4,8),
            l2_leaf_reg=trial.suggest_float('l2_leaf_reg',1,10,log=True))
    raise ValueError(f"no Optuna search space defined for '{name}' — add one in suggest()")

def build_tuned(name, params, strategy, seed=SEED):
    be = base_estimator(name, seed)
    be.set_params(**params)
    head = (MultiOutputClassifier(be) if strategy=='ovr'
            else ClassifierChain(be, order='random', random_state=seed))
    return Pipeline([('pre', make_preprocessor()), ('clf', head)])

def objective(trial):
    params = suggest(trial, TUNE_MODEL)
    mskf = MultilabelStratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    oof = np.zeros((len(X_train), len(LABELS)))
    for tr,va in mskf.split(X_train, y_train):
        m = build_tuned(TUNE_MODEL, params, TUNE_STRAT)
        m.fit(X_train.iloc[tr], y_train[tr])
        oof[va] = proba_matrix(m, X_train.iloc[va])
    thr = tune_thresholds(y_train, oof)
    return competition_score(y_train, apply_thr(oof, thr))[0]

study = optuna.create_study(direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print("best CV weighted-F1:", round(study.best_value,4))
print("best params:", study.best_params)
BEST_PARAMS = study.best_params

## 11 · Over-/under-fitting diagnostics
- **train↔CV gap** (from the sweep): big positive gap = overfit; both low = underfit.
- **Learning curve**: CV score vs training-set size — flat & low ⇒ more data won't help (underfit /
  noise ceiling); still rising ⇒ data-limited.

In [ ]:
# (a) train vs CV gap across the zoo
b2 = board.copy()
fig,ax=plt.subplots(figsize=(9,4))
x=np.arange(len(b2)); w=0.4
ax.bar(x-w/2, b2['train'], w, label='train'); ax.bar(x+w/2, b2['cv'], w, label='CV(OOF)')
ax.set_xticks(x); ax.set_xticklabels([f"{n}/{s}" for n,s in zip(b2['name'],b2['strategy'])],
    rotation=45, ha='right'); ax.legend(); ax.set_ylabel('weighted-F1')
ax.set_title('Train vs CV — gap reveals overfitting'); plt.tight_layout(); plt.show()

In [ ]:
# (b) learning curve for the tuned champion family
champ = build_tuned(TUNE_MODEL, BEST_PARAMS, TUNE_STRAT)
sizes=[0.2,0.4,0.6,0.8,1.0]; lc=[]
mskf=MultilabelStratifiedKFold(n_splits=3,shuffle=True,random_state=SEED)
for frac in sizes:
    fold=[]
    for tr,va in mskf.split(X_train,y_train):
        n=int(len(tr)*frac); tr=tr[:n]
        m=build_tuned(TUNE_MODEL,BEST_PARAMS,TUNE_STRAT); m.fit(X_train.iloc[tr],y_train[tr])
        p=proba_matrix(m,X_train.iloc[va]); t=tune_thresholds(y_train[va],p)
        fold.append(competition_score(y_train[va],apply_thr(p,t))[0])
    lc.append(np.mean(fold))
plt.figure(figsize=(6,4)); plt.plot([int(s*len(X_train)) for s in sizes], lc, 'o-')
plt.xlabel('training rows'); plt.ylabel('CV weighted-F1'); plt.title('Learning curve — champion')
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## 12 · Threshold tuning on `val.csv` (the official pattern, highest leverage)
Train the champion on **train**, `predict_proba` on **val**, pick the per-label F1-max threshold,
and measure the **val weighted-F1** — our honest proxy for the private 70% leaderboard.

In [ ]:
champion = build_tuned(TUNE_MODEL, BEST_PARAMS, TUNE_STRAT)
champion.fit(X_train, y_train)
val_proba = proba_matrix(champion, X_val)

thr_default = np.full(len(LABELS), 0.5)
thr_tuned   = tune_thresholds(y_val, val_proba)

s0,p0 = competition_score(y_val, apply_thr(val_proba, thr_default))
s1,p1 = competition_score(y_val, apply_thr(val_proba, thr_tuned))
print(f"VAL weighted-F1   @0.5 = {s0:.4f}   |   tuned = {s1:.4f}   (+{s1-s0:.4f})\n")
print(f"{'label':24s}{'thr':>6}{'F1@0.5':>9}{'F1@thr':>9}{'weight':>8}")
for j,l in enumerate(LABELS):
    print(f"{l:24s}{thr_tuned[j]:>6.2f}{p0[l]:>9.3f}{p1[l]:>9.3f}{CONFIG['weights'][l]:>8}")
FINAL_THR = thr_tuned

## 13 · Ensemble — blend top models (the real upgrade)
Average the **probabilities** of the strongest diverse models, retune thresholds on val. A blend
of decorrelated models usually beats any single one. Keep it only if val weighted-F1 improves.

In [ ]:
# Pick a few diverse strong performers (edit to taste)
BLEND = [(TUNE_MODEL, TUNE_STRAT)]
for cand in [('lgbm','ovr'),('catboost','ovr'),('rf','ovr'),('histgb','ovr')]:
    if cand not in BLEND and (cand[0] not in HAVE or HAVE.get(cand[0],True)):
        try: base_estimator(cand[0]); BLEND.append(cand)
        except Exception: pass
BLEND = BLEND[:3]
print("blending:", BLEND)

val_probas, test_probas, fitted = [], [], []
for name,strat in BLEND:
    m = (build_tuned(name, BEST_PARAMS, strat) if name==TUNE_MODEL else make_model(name,strat))
    m.fit(X_train, y_train); fitted.append((name,strat,m))
    val_probas.append(proba_matrix(m, X_val))
    test_probas.append(proba_matrix(m, X_test))

blend_val = np.mean(val_probas, axis=0)
thr_blend = tune_thresholds(y_val, blend_val)
s_blend,_ = competition_score(y_val, apply_thr(blend_val, thr_blend))
print(f"\nVAL weighted-F1 — champion={s1:.4f}  blend={s_blend:.4f}")

USE_BLEND = s_blend > s1
print("=> using", "BLEND" if USE_BLEND else "CHAMPION", "for submission")

## 14 · Build submission (+ format validation)
Apply the chosen model and tuned thresholds to **test**, align to `SampleSubmission` row order,
validate, and save. Rule-compliant: we submit binary labels and never train on `val`.

> *Optional upgrade for the final 2 submissions:* refit on **train+val** with the thresholds fixed,
> to use all labelled data. Left off by default to match the stated intended split.

In [ ]:
if USE_BLEND:
    test_proba = np.mean(test_probas, axis=0); thr_final = thr_blend; tag='blend'
else:
    test_proba = proba_matrix(champion, X_test); thr_final = FINAL_THR; tag=f'{TUNE_MODEL}_{TUNE_STRAT}'

pred = apply_thr(test_proba, thr_final)
sub = pd.DataFrame({ID_COL: test[ID_COL].values})
for j,l in enumerate(LABELS): sub[l] = pred[:,j]

# align to sample order & validate
sub = sample_sub[[ID_COL]].merge(sub, on=ID_COL, how='left')
assert sub[LABELS].isna().sum().sum()==0, "missing IDs vs sample submission!"
assert list(sub.columns)==[ID_COL]+LABELS
assert sub.shape[0]==sample_sub.shape[0]
assert set(np.unique(sub[LABELS].values)).issubset({0,1})

out = os.path.join(DATA_DIR, f'submission_{tag}.csv')
sub.to_csv(out, index=False)
print("saved:", out)
print("predicted positive-rates:\n", sub[LABELS].mean().round(3).to_string())

## 15 · Persist artifacts & submission log
Save the model + thresholds (so a Colab disconnect never loses work) and append a row to the
submission log — track **local val score vs the Zindi score** to confirm CV↔LB correlation and
stay inside the 25-submission budget.

In [ ]:
import joblib
stamp = time.strftime('%Y%m%d_%H%M%S')
joblib.dump({'model':(fitted if USE_BLEND else champion),'thresholds':thr_final.tolist(),
             'features':FEATURES,'labels':LABELS,'tag':tag},
            os.path.join(ART_DIR, f'model_{tag}_{stamp}.joblib'))

log_path = os.path.join(ART_DIR, 'submission_log.csv')
row = dict(time=stamp, model=tag, n_trials=N_TRIALS,
           val_score=round(s_blend if USE_BLEND else s1,4),
           thresholds=json.dumps([round(t,2) for t in thr_final]),
           zindi_public_score='')   # <-- fill in by hand after you submit
log = pd.concat([pd.read_csv(log_path), pd.DataFrame([row])], ignore_index=True) \
        if os.path.exists(log_path) else pd.DataFrame([row])
log.to_csv(log_path, index=False)
print("artifact + log saved to", ART_DIR)
print(log.tail(5).to_string(index=False))

---
### Next-iteration ideas (cheap, high-value)
1. **Target/frequency encoding** of profiles (the signal ≈ per-profile gap frequency) — fit inside CV.
2. **Per-label models** with label-specific features / class weights, focused on Water·Refuse·Educ (76% of score).
3. **Stacking**: feed OOF probabilities of base models into a meta-LogReg per label.
4. **Threshold stability**: average tuned thresholds over CV folds to avoid overfitting `val`.
5. **Document Gemini usage** here for the top-10 code review.
